# RSF Modelling
This notebook builds and evaluates a Random Survival Forest (RSF) model to predict when customers are likely to place their next order. It engineers behavioral features derived from purchase history, fits a baseline RSF, tunes hyperparameters via cross-validated grid search, and compares baseline against tuned performance on a held-out test set.

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

from sme_kt_zh_collaboration_forecasting import modelling as sm
from sme_kt_zh_collaboration_forecasting import utils as su

## Prepare the Input Data
The raw sales data is loaded and deduplicated here. This creates a clean event history that can safely be used for feature engineering, train-test splitting, and survival data construction.

In [ ]:
df = su.read_sales_data("../data/sales_df.csv")
len_b = len(df)
df = df.drop_duplicates()
len_a = len(df)

print(f"Dropped {len_b - len_a} duplicates.")

## Feature Engineering
Behavioral features are derived from the full transaction history before any train-test split, so that statistics like rolling order counts and average inter-purchase gaps use the maximum available history. Features capture both short-term activity momentum and long-term customer ordering cadence.

In [ ]:
df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["customer", "date"]).reset_index(drop=True)

# 1. Capped Cumulative Orders
df["num_prior_orders_capped"] = df.groupby("customer").cumcount().clip(upper=15)

# 2. Rolling Order Count (Momentum over the last 365 days)
df["dummy_counter"] = 1
df = df.set_index("date")
df["orders_last_365d"] = (
    df.groupby("customer")["dummy_counter"]
    .transform(lambda x: x.rolling("365D").count())
    .values
)
df = df.reset_index().drop(columns=["dummy_counter"])
df = df.sort_values(["customer", "date"]).reset_index(drop=True)

# 3. Days Since Previous Order
df["days_since_last_order"] = df.groupby("customer")["date"].diff().dt.days

# 4. Average Inter-Purchase Time (Historical Cadence)
df["avg_prior_gap"] = df.groupby("customer")["days_since_last_order"].transform(
    lambda x: x.expanding().mean()
)

# 5. Fill NA values for the very first order of every customer
global_median_gap = df["days_since_last_order"].median()
df["days_since_last_order"] = df["days_since_last_order"].fillna(global_median_gap)
df["avg_prior_gap"] = df["avg_prior_gap"].fillna(global_median_gap)

## Train / Test Split and Survival Data Preparation
The raw data is split at the cutoff date. The training cohort is restricted to customers with at least three orders to ensure enough history for the model. `prepare_data` then converts raw transactions into a censored survival table with a duration and event indicator per row.

In [ ]:
# 1. Split Raw Data
CUTOFF = "2024-12-31"
TOP_K = 50
df_train_raw, df_test_raw = sm.create_test_train(df, CUTOFF)

# 2. Initial Filtering
# We filter for >= 3 orders IN TRAIN to ensure we have enough history to model
df_train_raw = sm.filter_for_n_orders(df_train_raw, n=3)
train_customers = df_train_raw["customer"].unique()
df_test_raw = df_test_raw[df_test_raw["customer"].isin(train_customers)].copy()

# 3. Prepare Survival Tables
train_final = sm.prepare_data(df_train_raw, CUTOFF)
test_final = sm.prepare_data(df_test_raw, CUTOFF)

## Fit the RSF Model
The feature matrix is constructed by one-hot-encoding the customer category and aligning the test matrix to the training columns. The Random Survival Forest is then fitted on the prepared training data.

In [ ]:
MODEL_FEATURES = [
    "num_prior_orders_capped",
    "orders_last_365d",
    "days_since_last_order",
    "avg_prior_gap",
    "customer_cat",
]

train_final["event_bool"] = train_final["event"].astype(bool)
test_final["event_bool"] = test_final["event"].astype(bool)

y_train = Surv.from_dataframe("event_bool", "duration", train_final)

X_train = pd.get_dummies(
    train_final[MODEL_FEATURES], columns=["customer_cat"], drop_first=True
)
TRAIN_COLUMNS = X_train.columns

rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    n_jobs=-1,
    random_state=42,
)

print("Fitting Random Survival Forest...")
rsf.fit(X_train, y_train)

## Evaluate RSF Performance
The model is scored on the held-out test set using the concordance index and the top-k ranking recall. A comparison row is stored for the final summary table.

In [ ]:
X_test = pd.get_dummies(
    test_final[MODEL_FEATURES], columns=["customer_cat"], drop_first=True
)
X_test = X_test.reindex(columns=TRAIN_COLUMNS, fill_value=0)

rsf_c_index = sm.c_index_rsf(rsf, X_test, test_final)
prio = sm.predicted_vs_real_priorities_rsf(
    rsf, test_final, TRAIN_COLUMNS, MODEL_FEATURES, customer_col="customer"
)
rsf_ranking_summary = sm.summarize_top_k_predictions(prio, top_k=TOP_K)
rsf_comparison_row = sm.comparison_row(
    model_name="RSF",
    c_index=rsf_c_index,
    ranking_summary=rsf_ranking_summary,
)

print(f"Test Concordance Index (C-Index): {rsf_c_index:.4f}")
print(
    f"Within the top {TOP_K}, {rsf_ranking_summary['correct_predictions']} customers are in both lists."
)
print(f"Recall@{TOP_K}:    {rsf_ranking_summary['recall_at_k']:.2%}")

## Tune Hyperparameters
A grid search with three-fold cross-validation explores tree complexity and feature-sampling settings. The best estimator replaces the baseline model for the final comparison.

In [ ]:
print("Tuning RSF Hyperparameters...")
param_grid = {
    "n_estimators": [100, 250],
    "min_samples_leaf": [5, 15, 25],  # Lower = more complex trees
    "min_samples_split": [5, 10, 25],
    "max_features": ["sqrt", None],  # None = use all features at every split
}

# GridSearchCV natively uses concordance index for survival models in sksurv
grid_search = GridSearchCV(
    RandomSurvivalForest(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validated C-Index: {grid_search.best_score_:.4f}")

rsf_best = grid_search.best_estimator_

## Evaluate Tuned RSF and Compare
The tuned model is evaluated on the same test set using the same metrics. The final comparison table shows concordance and recall for the baseline and tuned RSF side by side.

In [ ]:
rsf_best_c_index = sm.c_index_rsf(rsf_best, X_test, test_final)
prio_best = sm.predicted_vs_real_priorities_rsf(
    rsf_best, test_final, TRAIN_COLUMNS, MODEL_FEATURES, customer_col="customer"
)
rsf_best_ranking_summary = sm.summarize_top_k_predictions(prio_best, top_k=TOP_K)
rsf_best_comparison_row = sm.comparison_row(
    model_name="RSF (tuned)",
    c_index=rsf_best_c_index,
    ranking_summary=rsf_best_ranking_summary,
)

print(f"Test Concordance Index (C-Index): {rsf_best_c_index:.4f}")
print(
    f"Within the top {TOP_K}, {rsf_best_ranking_summary['correct_predictions']} customers are in both lists."
)
print(f"Recall@{TOP_K}:    {rsf_best_ranking_summary['recall_at_k']:.2%}")

## Results Discussion: Tuned vs. Baseline RSF

While hyperparameter tuning slightly improved the **C-index**, we observed a simultaneous decline in **Recall**. This divergence often indicates high variance or that the model is capturing noise rather than generalizable patterns. 

**Key Takeaway:** Naive parameter optimization is rarely a silver bullet. This result highlights that real performance gains are usually found in **feature engineering** and data quality rather than just grid searching. We have intentionally left the model in this state to serve as a starting point for the reader to explore further data refinement and advanced feature construction.

In [ ]:
comparison_df = pd.DataFrame(
    [
        rsf_comparison_row,
        rsf_best_comparison_row,
    ]
)
comparison_df[["c_index", "recall_at_k"]] = comparison_df[
    [
        "c_index",
        "recall_at_k",
    ]
].round(4)

comparison_df